# Chapter 31: Cleaning, Transformation, and Reproducible Pipelines

This notebook turns inconsistent NRG order revisions into a traceable analysis-ready table.


In [1]:
import matplotlib.pyplot as plt
from datasciencebook.cleaning_pipeline import (
    normalize_text, parse_quantity, map_category, latest_revisions,
    numeric_profile, category_counts, transformation_record,
)


## 1. Preserve raw values and create cleaned fields


In [2]:
raw = [
    {'order_id': ' a-01 ', 'revision_number': 1, 'product': ' Instant  Noodle ', 'quantity': '1,000 units', 'city': 'JAKRTA'},
    {'order_id': 'A-01', 'revision_number': 2, 'product': 'instant noodle', 'quantity': '1,200', 'city': 'Jakarta'},
    {'order_id': 'B-02', 'revision_number': 1, 'product': ' Coffee Beans ', 'quantity': '500 units', 'city': 'DUBAI'},
    {'order_id': 'C-03', 'revision_number': 1, 'product': 'Spice Mix', 'quantity': '-', 'city': 'Surabaya'},
]
city_map = {'jakrta': 'Jakarta', 'jakarta': 'Jakarta', 'dubai': 'Dubai', 'surabaya': 'Surabaya'}
cleaned = []
for row in raw:
    cleaned.append({
        **row,
        'order_id_raw': row['order_id'],
        'order_id': normalize_text(row['order_id'], case='upper'),
        'product_clean': normalize_text(row['product']),
        'quantity_clean': parse_quantity(row['quantity']),
        'city_clean': map_category(row['city'], city_map),
    })
for row in cleaned:
    print(row['order_id'], row['product_clean'], row['quantity_clean'], row['city_clean'])


A-01 instant noodle 1000.0 Jakarta
A-01 instant noodle 1200.0 Jakarta
B-02 coffee beans 500.0 Dubai
C-03 spice mix None Surabaya


## 2. Select revisions deterministically


In [3]:
final = latest_revisions(cleaned)
print('Input rows:', len(cleaned))
print('Final rows:', len(final))
print('Final order IDs:', [row['order_id'] for row in final])
print('Quantity profile:', numeric_profile([row['quantity_clean'] for row in final]))


Input rows: 4
Final rows: 3
Final order IDs: ['A-01', 'B-02', 'C-03']
Quantity profile: {'count': 2, 'minimum': 500.0, 'mean': 850.0, 'maximum': 1200.0}


## 3. Compare categories before and after standardisation


In [4]:
print('Raw city counts:', category_counts([row['city'] for row in raw]))
print('Clean city counts:', category_counts([row['city_clean'] for row in final]))
log = transformation_record('select latest revision', len(cleaned), len(final), len(cleaned) - len(final))
print('Ledger:', log)


Raw city counts: {'DUBAI': 1, 'JAKRTA': 1, 'Jakarta': 1, 'Surabaya': 1}
Clean city counts: {'Dubai': 1, 'Jakarta': 1, 'Surabaya': 1}
Ledger: {'step': 'select latest revision', 'input_rows': 4, 'output_rows': 3, 'affected_values': 1, 'status': 'passed'}


## 4. Visualise the effect of cleaning


In [5]:
labels = ['Raw rows', 'Final rows', 'Raw city labels', 'Clean city labels']
values = [len(raw), len(final), len(set(row['city'] for row in raw)), len(set(row['city_clean'] for row in final))]
plt.bar(labels, values, color=['#4472C4', '#70AD47', '#4472C4', '#70AD47'])
plt.ylabel('Count')
plt.title('NRG before and after controlled cleaning')
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()


<Figure size 640x480 with 1 Axes>

## 5. Confirm idempotent text cleaning


In [6]:
once = normalize_text('  Instant   Noodle ')
twice = normalize_text(once)
print('First pass:', once)
print('Second pass:', twice)
print('Idempotent:', once == twice)


First pass: instant noodle
Second pass: instant noodle
Idempotent: True


## Practice

Add one invalid quantity, capture the parsing failure without changing the raw value, and create a warning ledger entry.


In [ ]:
# Implement the practice step here.
